# 19. 고객 세분화 군집 분석 (Mall Customers Clustering)

`Mall_Customers.csv` 데이터를 표준화(`StandardScaler`)한 뒤 엘보우 기법과 `KMeans(n_clusters=5)`로 고객 군집을 세분화하고 군집별 평균 특성과 소비 패턴을 시각화합니다.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

candidates = [
    Path('Study_Python/[7] Machine_Learning/data/Mall_Customers.csv'),
    Path('../[7] Machine_Learning/data/Mall_Customers.csv'),
    Path('data/Mall_Customers.csv'),
]
csv_path = next((p for p in candidates if p.is_file()), None)
if csv_path is None:
    raise FileNotFoundError('Mall_Customers.csv 파일을 찾을 수 없습니다.')

df = pd.read_csv(csv_path)
df = df.drop(columns=['CustomerID'])
print(df.head())

In [ ]:
# 수치형 특성 스케일링 (Age, Annual Income (k$), Spending Score (1-100))
features = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df[features])

# 엘보우 기법으로 최적 군집 수 확인
inertia = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(scaled_data)
    inertia.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(range(1, 11), inertia, marker='o')
plt.title('Elbow Method (Mall Customers)')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# KMeans(n_clusters=5) 학습 및 군집별 평균 분석
kmeans = KMeans(n_clusters=5, n_init=10, random_state=42)
df['Cluster_ID'] = kmeans.fit_predict(scaled_data)

cluster_summary = df.select_dtypes(include=['number']).groupby('Cluster_ID').mean().round(2)
print(cluster_summary)

In [ ]:
# 군집별 산점도 시각화 (연간 소득 vs 소비 점수)
plt.figure(figsize=(8, 5))
sns.scatterplot(x='Annual Income (k$)', y='Spending Score (1-100)', hue='Cluster_ID', data=df, palette='coolwarm', s=80)
plt.title('Annual Income vs Spending Score by Cluster')
plt.show()